# Exploratory Data Analysis for Pairs Trading

This notebook demonstrates how to load and explore financial data for pairs trading analysis.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime

# Import project modules
import sys
sys.path.append('../..')
from src.pairs_selection import find_cointegrated_pairs, test_cointegration
from src.utils import calculate_returns

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

Download historical price data for a set of stocks.

In [ ]:
# Define parameters
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']
start_date = '2020-01-01'
end_date = '2023-12-31'

# Download data
data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']
print(f"Downloaded {len(data)} days of data for {len(tickers)} stocks")
data.head()

## 2. Visualize Price Series

In [ ]:
# Normalize prices for comparison
normalized_data = data / data.iloc[0] * 100

# Plot
plt.figure(figsize=(14, 6))
for ticker in tickers:
    plt.plot(normalized_data.index, normalized_data[ticker], label=ticker, linewidth=2)

plt.title('Normalized Price Series (Base = 100)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Normalized Price', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Calculate Correlation Matrix

In [ ]:
# Calculate correlation
correlation_matrix = data.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Stock Price Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Find Cointegrated Pairs

In [ ]:
# Find cointegrated pairs
pairs = find_cointegrated_pairs(data, significance_level=0.05)

print(f"Found {len(pairs)} cointegrated pairs:\n")
for stock1, stock2, p_value in pairs:
    print(f"{stock1} - {stock2}: p-value = {p_value:.4f}")

## 5. Analyze a Specific Pair

In [ ]:
# Select the first pair
if len(pairs) > 0:
    stock1, stock2, p_value = pairs[0]
    
    # Calculate spread
    spread = data[stock1] - data[stock2]
    
    # Calculate z-score
    rolling_mean = spread.rolling(window=20).mean()
    rolling_std = spread.rolling(window=20).std()
    zscore = (spread - rolling_mean) / rolling_std
    
    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Price series
    axes[0].plot(data.index, data[stock1], label=stock1, linewidth=2)
    axes[0].plot(data.index, data[stock2], label=stock2, linewidth=2)
    axes[0].set_title(f'{stock1} vs {stock2} Price Series', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Price ($)', fontsize=11)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Spread
    axes[1].plot(spread.index, spread, label='Spread', color='purple', linewidth=2)
    axes[1].plot(rolling_mean.index, rolling_mean, label='20-day MA', 
                 color='orange', linestyle='--', linewidth=2)
    axes[1].set_title('Spread', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Spread ($)', fontsize=11)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Z-score
    axes[2].plot(zscore.index, zscore, label='Z-score', color='green', linewidth=2)
    axes[2].axhline(y=2, color='r', linestyle='--', label='Entry threshold (+2)')
    axes[2].axhline(y=-2, color='r', linestyle='--', label='Entry threshold (-2)')
    axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[2].set_title('Spread Z-Score', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Z-Score', fontsize=11)
    axes[2].set_xlabel('Date', fontsize=11)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nStatistics for {stock1}-{stock2} pair:")
    print(f"Correlation: {data[[stock1, stock2]].corr().iloc[0, 1]:.4f}")
    print(f"Cointegration p-value: {p_value:.4f}")
    print(f"Mean spread: {spread.mean():.2f}")
    print(f"Spread volatility: {spread.std():.2f}")
else:
    print("No cointegrated pairs found. Try adjusting the significance level or using different stocks.")

## 6. Summary Statistics

In [ ]:
# Calculate returns
returns = calculate_returns(data)

# Summary statistics
summary = pd.DataFrame({
    'Mean Return': returns.mean(),
    'Volatility': returns.std(),
    'Sharpe Ratio': returns.mean() / returns.std() * np.sqrt(252),
    'Min': returns.min(),
    'Max': returns.max()
})

print("\nSummary Statistics:")
print(summary)

## Next Steps

1. Explore additional pairs with different significance levels
2. Implement feature engineering for ML models
3. Develop trading signals and backtest strategies
4. Train machine learning models for signal prediction